# Entrenamiento YOLOv8 — Logo Placement
**Corre este notebook en Google Colab con GPU T4 (gratis)**

### Pasos antes de abrir Colab:
1. Comprime la carpeta `dataset/` en un zip: `dataset.zip`
2. Sube `dataset.zip` a tu Google Drive
3. Abre este notebook en Colab: `Archivo → Abrir notebook → Google Drive`
4. Cambia el runtime a GPU: `Entorno de ejecución → Cambiar tipo → T4 GPU`


In [ ]:
# Paso 1: Montar Google Drive y descomprimir dataset
from google.colab import drive
drive.mount('/content/drive')

import zipfile, os

# Ajusta esta ruta a donde subiste dataset.zip en tu Drive
ZIP_PATH = '/content/drive/MyDrive/dataset.zip'
EXTRACT_TO = '/content/proyecto_vision'

os.makedirs(EXTRACT_TO, exist_ok=True)

with zipfile.ZipFile(ZIP_PATH, 'r') as zf:
    zf.extractall(EXTRACT_TO)

print('Dataset descomprimido en', EXTRACT_TO)

In [ ]:
# Paso 2: Instalar dependencias
!pip install ultralytics -q

In [ ]:
# Paso 3: Verificar que la GPU está disponible
import torch
print('CUDA disponible:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))

In [ ]:
# Paso 4: Crear data.yaml con rutas correctas para Colab
import yaml

DATASET_PATH = f'{EXTRACT_TO}/dataset'

data_config = {
    'path': DATASET_PATH,
    'train': 'images/train',
    'val': 'images/val',
    'nc': 2,
    'names': ['bottles', 'tshirts']
}

yaml_path = f'{DATASET_PATH}/data_colab.yaml'
with open(yaml_path, 'w') as f:
    yaml.dump(data_config, f)

print('data.yaml creado:', yaml_path)

In [ ]:
# Paso 5: Entrenar YOLOv8n con GPU
from ultralytics import YOLO

model = YOLO('yolov8n.pt')

results = model.train(
    data=yaml_path,
    epochs=150,
    imgsz=640,
    batch=16,      # Con GPU T4 se puede usar batch=16 o 32
    device=0,      # GPU
    workers=2,
    patience=30,
    optimizer='AdamW',
    lr0=0.001,
    mosaic=1.0,
    mixup=0.1,
    fliplr=0.5,
    project='/content/runs/train',
    name='logo_placement',
    exist_ok=True,
    plots=True,
)

print('mAP50   :', results.results_dict.get('metrics/mAP50(B)'))
print('mAP50-95:', results.results_dict.get('metrics/mAP50-95(B)'))

In [ ]:
# Paso 6: Guardar los pesos en Google Drive
import shutil

SAVE_DIR = '/content/drive/MyDrive/logo_placement_model'
shutil.copytree('/content/runs/train/logo_placement', SAVE_DIR, dirs_exist_ok=True)
print('Pesos guardados en Drive:', SAVE_DIR)

In [ ]:
# Paso 7: Evaluar en el set de validación
best_model = YOLO('/content/runs/train/logo_placement/weights/best.pt')
val_results = best_model.val(data=yaml_path, device=0)

print('Precisión  :', val_results.results_dict.get('metrics/precision(B)'))
print('Recall     :', val_results.results_dict.get('metrics/recall(B)'))
print('mAP50      :', val_results.results_dict.get('metrics/mAP50(B)'))